# 第1回 練習問題 解答例

先に準備のセルを実行してください（本編と同じ道具を作ります）。

In [ ]:
# このセルは毎回最初に実行します（必要な道具を読み込む）
import numpy as np                      # 数値計算（ルートや三角関数など）
import matplotlib.pyplot as plt         # グラフを描く
from IPython.display import display     # 図を何枚も表示するため

import qiskit
import qiskit_aer
from qiskit import QuantumCircuit                       # 量子回路を作る
from qiskit.quantum_info import Statevector             # 量子ビットの「状態」を表す
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from qiskit_aer import AerSimulator                     # パソコンの中のシミュレーター
from qiskit_aer.primitives import SamplerV2             # 「測定して結果を数える」道具
from qiskit.transpiler import generate_preset_pass_manager

print("Qiskit:", qiskit.__version__, "/ Aer:", qiskit_aer.__version__)

In [ ]:
simulator = AerSimulator()
# 回路をシミュレーターが実行できる形に変換する「変換器」
pm = generate_preset_pass_manager(backend=simulator, optimization_level=1)
sampler = SamplerV2()


def measure(state, shots):
    """量子ビットを state の状態にして、shots 回 測定した結果を数えて返す"""
    qc = QuantumCircuit(1)
    qc.prepare_state(state, 0)   # 量子ビット0番を、指定した状態にする
    qc.measure_all()             # 測定する
    job = sampler.run([pm.run(qc)], shots=shots)
    return job.result()[0].data.meas.get_counts()


def probs(state):
    """測定したとき 0 と 1 が出る確率を、見やすく(小数3けたで)表示する"""
    return {str(k): round(float(v), 3) for k, v in state.probabilities_dict().items()}


def state_from_angle(theta_deg):
    """ブロッホ球の北極から theta_deg 度 傾けた状態を作る"""
    t = np.radians(theta_deg)
    return Statevector([np.cos(t / 2), np.sin(t / 2)])

plus = Statevector([1 / np.sqrt(2), 1 / np.sqrt(2)])

## 問題1: 70% / 30% の状態

確率は「数の2乗」なので、$\alpha^2 = 0.7,\ \beta^2 = 0.3$ となる数、つまり
$\alpha = \sqrt{0.7} \approx 0.837,\ \beta = \sqrt{0.3} \approx 0.548$ にします。
（0.7 と 0.3 をそのまま入れると、確率が 0.49 と 0.09 になり、合計が1にならないので間違いです。）

In [ ]:
s70 = Statevector([np.sqrt(0.7), np.sqrt(0.3)])
print("(a) 確率:", probs(s70))

counts = measure(s70, shots=1000)
print("(b) 1000回測定:", counts)
plot_histogram(counts, title="0: 70%, 1: 30%")

0 が 700回前後、1 が 300回前後になればOKです（ぴったり700にはならず、毎回少しずれます）。

## 問題2: $\begin{pmatrix} 0.6 \\ 0.8 \end{pmatrix}$

(a) 手計算: $0.6^2 + 0.8^2 = 0.36 + 0.64 = 1$ なので、正しい状態です。

(b) 予想: 0 が出る確率 $0.6^2 = 0.36$（36%）、1 が出る確率 $0.8^2 = 0.64$（64%）。
2000回なら、0 が約720回、1 が約1280回のはずです。

(c) 1 の方が出やすいので、**南半球**（赤道より少し下）にあるはずです。

In [ ]:
s = Statevector([0.6, 0.8])
print("(a) is_valid:", s.is_valid())
print("(b) 理論の確率:", probs(s))

counts = measure(s, shots=2000)
print("    2000回測定:", counts)
display(plot_histogram(counts, title="(0.6, 0.8)"))
display(plot_bloch_multivector(s, title="(0.6, 0.8)", figsize=(3, 3)))

矢印が赤道より少し下（南半球）を向いていれば予想どおりです。
ちなみに北極からの角度は $\cos^2(\theta/2) = 0.36$ より $\theta \approx 106^\circ$ で、90°（赤道）より少しだけ南です。

## 問題3: 測定回数とばらつき

In [ ]:
for shots in [100, 10000]:
    ratios = []
    for trial in range(5):
        counts = measure(plus, shots=shots)
        ratios.append(counts.get("0", 0) / shots)
    print(f"{shots:5d}回 測定 ×5: {[round(r, 3) for r in ratios]}")
    print(f"      ばらつき(最大-最小) = {max(ratios) - min(ratios):.3f}\n")

100回のときは 0.43〜0.57 くらいまでばらつくことがありますが、10000回では 0.49〜0.51 くらいにおさまります
（実行するたびに数字は変わります）。

**言葉でのまとめ:** 測定の結果は毎回偶然で決まるので、少ない回数では本当の確率（0.5）からずれやすい。
**測定回数を増やすほど、割合は本当の確率に近づき、ばらつきが小さくなる。** だから確率を正確に知りたいときは、
測定回数（shots）を多くする必要がある。

**もう少し詳しく（発展）:** ばらつきの大きさはおよそ $1/\sqrt{\text{回数}}$ に比例します。
回数を100倍にすると、ばらつきは $1/\sqrt{100} = 1/10$ になります。上の結果でも、だいたい10分の1になっているはずです。
本物の量子コンピュータでも、同じ回路を何千回も実行して確率を調べるのはこのためです。